In [7]:
from jax.scipy.stats import norm as jnorm
import jax.numpy as jnp
from jax import grad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
plt.style.use("seaborn-v0_8")

In [23]:
prices_raw_0 = pd.read_csv("prices_round_3_day_0.csv", sep = ";", index_col = "timestamp")
prices_raw_1 = pd.read_csv("prices_round_4_day_1.csv", sep = ";", index_col = "timestamp")
prices_raw_2 = pd.read_csv("prices_round_4_day_2.csv", sep = ";", index_col = "timestamp")
prices_raw_3 = pd.read_csv("prices_round_4_day_3.csv", sep = ";", index_col = "timestamp")

FileNotFoundError: [Errno 2] No such file or directory: 'prices_round_4_day_1.csv'

In [9]:
def black_scholes(S, K, T, r, sigma, q=0, otype="call"):
  d1 = (jnp.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * jnp.sqrt(T))
  d2 = d1 - sigma * jnp.sqrt(T)
  if otype == "call":
    call = S * jnp.exp(-q * T) * jnorm.cdf(d1, 0, 1) - K * jnp.exp(-r * T) * jnorm.cdf(d2, 0, 1)
    return call 
  else:
    put = K * jnp.exp(-r * T) * jnorm.cdf(-d2, 0, 1) - S * jnp.exp(-q * T) * jnorm.cdf(-d1, 0, 1)
    return put 

In [10]:
def loss(S, K, T, r, sigma_guess, price, q=0, otype="call"):

  theoretical_price = black_scholes(S, K, T, r, sigma_guess, q, otype)

  market_price = price

  return theoretical_price - market_price

loss_grad = grad(loss, argnums=4)

In [11]:
vev

,product,mid_price
timestamp,,
0,VEV_5400,23.0
0,VEV_6500,0.5
0,VEV_5500,8.5
0,VEV_5200,101.5
0,VEV_5300,53.0
...,...,...
999900,VEV_4000,1244.0
999900,VEV_5100,164.0
999900,VEV_6000,0.5


In [12]:
def solve_for_iv(S, K, T, r, price, sigma_guess = 3.0, q=0, otype="call",
                N_iter = 20, epsilon = 0.001, verbose = True):
  sigma = sigma_guess
  for i in range(N_iter):
    print("\nIteration: ", i)

    loss_val = loss(S, K, T, r, sigma, price, q, otype)

    if verbose:
      print("Current Error in Theoretical vs Market Price:")
      print(loss_val)

    if abs(loss_val) < epsilon:
      break
    else:
      loss_grad_val = loss_grad(S, K, T, r, sigma, price, q, otype)

      sigma = sigma - loss_val / loss_grad_val

  return sigma

# calculated_implied_vol = solve_for_iv(S, K, T, r, price)
# print("------------------------------------")
# print("Optimised Implied Volatility: ", calculated_implied_vol)

In [13]:
def log_moneyness(S, K):
    return jnp.log(S / K)

In [14]:
# vev has index = timestamp, columns include: product, mid_price
# first move timestamp out of index so we can pivot on it
vev_wide = (
    vev.reset_index()  # makes timestamp a normal column (usually named "timestamp")
       .pivot(index="timestamp", columns="product", values="mid_price")
       .sort_index()
)

In [15]:
vev_wide_4000 = vev_wide[["VELVETFRUIT_EXTRACT", "VEV_4000"]]
vev_wide_4000

product,VELVETFRUIT_EXTRACT,VEV_4000
timestamp,,
0,5250.0,1250.0
100,5250.5,1250.5
200,5250.5,1250.5
300,5250.5,1250.5
400,5250.5,1250.5
...,...,...
999500,5244.5,1244.5
999600,5244.0,1245.5
999700,5245.5,1245.5


In [16]:
S = 5250.0
K = 4000.0
T = 5.0
r = 0.0
sigma = 0.1
q = 0
otype = "call"

price = black_scholes(S, K, T, r, sigma, q, otype)
print(f"Theoretical Price: {price:.2f}")

Theoretical Price: 1305.38


In [17]:
price = 1250.0
calculated_implied_vol = solve_for_iv(S, K, T, r, price, 0.03)
print("------------------------------------")
print("Optimised Implied Volatility: ", calculated_implied_vol)


Iteration:  0
Current Error in Theoretical vs Market Price:
0.0017089844

Iteration:  1
Current Error in Theoretical vs Market Price:
0.0007324219
------------------------------------
Optimised Implied Volatility:  0.028451687


In [18]:
# vev_wide_4000["log_moneyness"] = np.log(vev_wide_4000["VELVETFRUIT_EXTRACT"] / 4000)
# vev_wide_4000["iv"] = np.nan
# for t in vev_wide.index:
#     S = vev_wide.loc[t, "VELVETFRUIT_EXTRACT"]
#     K = 4000
#     T = 5.0
#     r = 0.0
#     price = vev_wide.loc[t, "VEV_4000"]
#     iv = solve_for_iv(S, K, T, r, price, 0.03, verbose=False)
#     vev_wide_4000.loc[t, "iv"] = iv


In [19]:
vev_wide

product,VELVETFRUIT_EXTRACT,VEV_4000,VEV_4500,VEV_5000,VEV_5100,VEV_5200,VEV_5300,VEV_5400,VEV_5500,VEV_6000,VEV_6500
timestamp,,,,,,,,,,,
0,5250.0,1250.0,750.0,257.0,171.5,101.5,53.0,23.0,8.5,0.5,0.5
100,5250.5,1250.5,751.0,258.0,172.0,102.5,53.0,23.5,8.5,0.5,0.5
200,5250.5,1250.5,750.0,257.0,172.0,102.5,53.0,23.5,8.5,0.5,0.5
300,5250.5,1250.5,750.0,257.5,172.0,102.5,53.0,23.5,8.5,0.5,0.5
400,5250.5,1250.5,750.0,257.0,172.0,102.5,53.0,23.5,8.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...
999500,5244.5,1244.5,745.0,251.0,165.0,95.5,47.0,16.5,7.5,0.5,0.5
999600,5244.0,1245.5,746.0,252.0,166.0,96.5,47.0,16.5,7.5,0.5,0.5
999700,5245.5,1245.5,745.0,252.0,165.0,95.5,47.0,16.5,7.5,0.5,0.5


In [ ]:
vev_wide.VELVETFRUIT_EXTRACT.mean()
vev_wide.VELVETFRUIT_EXTRACT.std()

np.float64(5246.51025)

In [21]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.cm as cm

STRIKES = {
    'VEV_4000': 4000, 'VEV_4500': 4500, 'VEV_5000': 5000,
    'VEV_5100': 5100, 'VEV_5200': 5200, 'VEV_5300': 5300,
    'VEV_5400': 5400, 'VEV_5500': 5500, 'VEV_6000': 6000,
    'VEV_6500': 6500
}

# --- BS functions (same as before) ---
def bs_price(S, K, T, sigma, r=0):
    if T <= 0: return max(S - K, 0)
    d1 = (np.log(S/K) + 0.5*sigma**2*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_vega(S, K, T, sigma):
    if T <= 0: return 0.0
    d1 = (np.log(S/K) + 0.5*sigma**2*T) / (sigma*np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)

def compute_iv(market_price, S, K, T, tol=1e-5, max_iter=50):
    intrinsic = max(S - K, 0.0)
    if market_price <= intrinsic + 1e-6 or T <= 0:
        return np.nan
    sigma = 0.3
    for _ in range(max_iter):
        p  = bs_price(S, K, T, sigma)
        v  = bs_vega(S, K, T, sigma)
        if v < 1e-10: return np.nan
        sigma -= (p - market_price) / v
        sigma  = np.clip(sigma, 0.001, 10.0)
        if abs(p - market_price) < tol: break
    return sigma if 0.01 < sigma < 5.0 else np.nan

# ── Assume df has:
#    index = timestamp (0..39999 or a day-aware index)
#    columns = ['VELVETFRUIT_EXTRACT', 'VEV_4000', ..., 'VEV_6500']
#    4 days of data → timestamps_per_day = len(df) // 4

TIMESTAMPS_PER_DAY = len(vev_wide) // 4
START_TTE = 8  # day 0 of your data = TTE 8

def ts_to_tte(ts):
    day = int(ts) // TIMESTAMPS_PER_DAY      # 0,1,2,3
    return (START_TTE - day) / 365

# ── Melt wide → long
rows = []
for voucher, K in STRIKES.items():
    tmp = vev_wide[['VELVETFRUIT_EXTRACT', voucher]].copy()
    tmp.columns = ['S', 'market_price']
    tmp['K'] = K
    tmp['voucher'] = voucher
    tmp['T'] = [ts_to_tte(ts) for ts in vev_wide.index]
    rows.append(tmp)

long = pd.concat(rows).dropna(subset=['S', 'market_price'])

# ── Compute IV (slow but readable — vectorise later if needed)
long['iv'] = [
    compute_iv(row.market_price, row.S, row.K, row.T)
    for row in long.itertuples()
]

long['log_moneyness'] = np.log(long['K'] / long['S'])
long = long.dropna(subset=['iv'])
long = long[(long['iv'] > 0.01) & (long['iv'] < 5.0)]

In [22]:
print(vev_wide[['VELVETFRUIT_EXTRACT'] + list(STRIKES.keys())].describe())

product  VELVETFRUIT_EXTRACT      VEV_4000     VEV_4500      VEV_5000  \
count           10000.000000  10000.000000  10000.00000  10000.000000   
mean             5246.510250   1246.524100    746.52160    253.259500   
std                13.677795     13.698111     13.68664     12.488839   
min              5216.500000   1215.000000    716.50000    226.000000   
25%              5236.500000   1236.500000    736.50000    244.000000   
50%              5244.500000   1244.500000    745.00000    251.000000   
75%              5255.500000   1255.500000    755.50000    261.000000   
max              5284.500000   1285.000000    785.00000    289.000000   

product      VEV_5100      VEV_5200      VEV_5300      VEV_5400      VEV_5500  \
count    10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean       168.107700     97.470900     48.892500     18.467250      8.058950   
std         10.954784      8.165664      5.179954      2.780724      1.236892   
min        145.000